# Variable speed of light and the horizon problem

*cosmology · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/cosmology/variable_speed_of_light)

Why is the CMB the same temperature in every direction when, in standard ΛCDM, its sky splits into about $10^4$ patches that never exchanged a signal? Integrating $\chi_p = \int c\,d\ln a/(aH)$ for a batch of universes compares three answers: none (ΛCDM), inflation ($\sim 60$ e-folds of a shrinking Hubble radius) and the speculative variable speed of light (VSL) idea of Moffat and Albrecht–Magueijo, where light was some $10^{26}$ times faster before a phase transition. Inflation is the mainstream explanation; VSL is a minority, speculative alternative.

**Vector form**

$$\chi_p(a) = \int_0^{a}\frac{c(a')\,da'}{a'^2H(a')},\qquad H = H_0\sqrt{\Omega_r a^{-4} + \Omega_m a^{-3} + \Omega_\Lambda},\qquad c(a) = c_0\Big[1 + (r-1)\,\sigma\Big(\frac{\ln a_* - \ln a}{w}\Big)\Big],\qquad \theta = \frac{\chi_p(a_\mathrm{rec})}{D},\quad N_\mathrm{patch} = \frac{2}{1-\cos\theta}$$

**Index (tensor) form**

$$r^H_{ma} = \frac{c_{ma}}{a_a H_{ma}},\qquad \chi_{ma} = \sum_{a'<a}\tfrac12\big(r^H_{m,a'} + r^H_{m,a'+1}\big)\,\Delta x,\qquad D = \chi_{m,\mathrm{today}} - \chi_{m,\mathrm{rec}},\qquad \text{solved} \iff \chi_{m,\mathrm{rec}} \ge D$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x_a = \ln a_a$ | `log_a_grid(a_min, n)` | (A,) | grid in $\ln a$ from far before $a_*$ to today |
| $c_{ma}$ | `speed_of_light(a, c_ratio, a_star)` | (M, A) | speed of light of universe (frame) $m$ at every $a$: $r$ times $c_0$ before $a_*$ |
| $H_{ma}$ | `hubble_with_inflation(a, efolds, a_star)` | (M, A) | expansion rate, frozen at $H(a_*)$ during $N$ e-folds of inflation |
| $r^H_{ma}$ | `c / (a * H)` | (M, A) | comoving Hubble radius: how far light gets in one expansion time |
| $\chi_{ma}$ | `particle_horizon(x, r_H)` | (M, A) | cumulative trapezoid along $a$: the comoving particle horizon |
| $\theta_m,\ N_\mathrm{patch}$ | `sky_statistics(x, h)` | (M,) | angle of one causal patch on the CMB sky and the number of patches |

Planck 2018 parameters ($H_0 = 67.4$ km/s/Mpc, $\Omega_m = 0.315$, $\Omega_r = 9.1\times10^{-5}$, flat), last scattering at $a_\mathrm{rec} = 1/1090$. ΛCDM gives a particle horizon of 14.2 Gpc today but only about 0.28 Gpc at recombination, so a causal patch spans about 1.2° and the sky holds about $10^4$ of them. Light moves at 45° in the conformal diagram; the past light cones of two opposite CMB points overlap only if $\chi_p(a_\mathrm{rec}) \ge D$ (distance to the CMB, 13.9 Gpc). Inflation does this by $N$ e-folds of de Sitter expansion ending at $a_*$ (the comoving Hubble radius shrinks $\propto 1/a$); VSL keeps the standard expansion but makes $c$ larger by $r$ before a transition at $a_*$, a smooth step of width $w = 0.1$ e-folds. The animation turns both on gradually. Status: VSL (Moffat 1993; Albrecht and Magueijo 1999) is speculative and a minority view; it needs a frame in which a dimensional constant can vary (only dimensionless combinations such as $\alpha = e^2/4\pi\varepsilon_0\hbar c$ are measurable), its predictions are model dependent, and it has no observational support. Inflation is the standard explanation, also unconfirmed directly (no primordial B-modes yet) but consistent with the measured spectrum of CMB fluctuations. Bounds (printed in the terminal): atomic clocks $\dot\alpha/\alpha = (1.0\pm1.1)\times10^{-18}$/yr, the Oklo reactor $|\Delta\alpha/\alpha| \lesssim 10^{-7}$ over 2 Gyr, the CMB $|\Delta\alpha/\alpha| \lesssim 4\times10^{-3}$ at recombination, and GW170817 $-3\times10^{-15} \le (v_\mathrm{gw}-c)/c \le 7\times10^{-16}$. A transition at $a_* \sim 10^{-28}$ evades all of them, which is also why it is so hard to test.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/cosmology/variable_speed_of_light/main.py
"""Main loop: parameters → tensors → solve → plot."""

import math
from pathlib import Path

from problems._common.plotting import frame_indices
from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    a_star = 10 ** params["log10_a_star"]
    x = tensors.log_a_grid(min(a_star * math.exp(-params["efolds"] - 3), 1e-40), params["n_grid"], dev)
    n_frames = len(frame_indices(60, 60))
    c_ratio = 10 ** tensors.sweep(params["log10_c_ratio"], n_frames, dev)  # (F,) VSL switched on gradually
    efolds = tensors.sweep(params["efolds"], n_frames, dev)  # (F,) inflation switched on gradually

    std = solve.horizons(x, "lcdm")  # (1, A)
    vsl = solve.horizons(x, "vsl", c_ratio=c_ratio, a_star=a_star)  # (F, A): one universe per frame
    infl = solve.horizons(x, "inflation", efolds=efolds, a_star=a_star)  # (F, A)
    s_std, s_vsl, s_inf = (solve.sky_statistics(x, h) for h in (std, vsl, infl))

    D = float(s_std["distance"][0])
    print(f"Planck 2018 ΛCDM: particle horizon today χ_p(1) = {float(std['chi'][0, -1]) / 1e3:.2f} Gpc, "
          f"distance to last scattering D = {D / 1e3:.2f} Gpc")  # fmt: skip
    print(f"horizon at recombination χ_p(a_rec) = {float(s_std['chi_rec'][0]):.0f} Mpc → θ = "
          f"{math.degrees(float(s_std['theta'][0])):.2f}°, ≈ {float(s_std['patches'][0]):.0f} "
          "causally disconnected patches")  # fmt: skip
    r_min = solve.vsl_minimum_ratio(x, a_star)
    n_min = solve.inflation_minimum_efolds(x, a_star)
    print(f"transition at a_* = 1e{params['log10_a_star']:.0f}: needs c_early/c₀ ≥ {r_min:.2e} (VSL) "
          f"or N ≥ {n_min:.1f} e-folds (inflation)")  # fmt: skip
    for name, s, val in (("VSL", s_vsl, f"c_early/c₀ = {float(c_ratio[-1]):.1e}"),
                         ("inflation", s_inf, f"N = {float(efolds[-1]):.0f}")):  # fmt: skip
        print(
            f"{name:9s} {val}: χ_p(a_rec)/D = {float(s['chi_rec'][-1] / s['distance'][-1]):.3g}  "
            f"→ horizon problem {'solved' if bool(s['solved'][-1]) else 'NOT solved'}"
        )
    print("observational bounds (VSL here only changes c before a_*, far earlier than any of them probe):")
    for name, b in equation.BOUNDS.items():
        print(f"  {name:38s} {b['value'][0]:+.1e} … {b['value'][1]:+.1e} {b['unit']:14s} ({b['ref']})")

    return plot.animate(x, std, vsl, infl, (s_std, s_vsl, s_inf), c_ratio, efolds, a_star,
                        out_dir / "variable_speed_of_light.gif")  # fmt: skip


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/cosmology/variable_speed_of_light/equation.py
r"""Physics → tensor translation: horizons with a time-dependent speed of light.

Light travels a comoving distance dχ = c(t) dt / a. With x = ln a and dt = dx / H, the particle horizon (the farthest
comoving distance any signal can have covered since the beginning) and the comoving Hubble radius are

    χ_p(a) = ∫ c(a′) / (a′ H(a′)) d ln a′,          r_H(a) = c(a) / (a H(a)),
    H(a) = H₀ √(Ω_r a⁻⁴ + Ω_m a⁻³ + Ω_Λ)   (standard ΛCDM expansion).

The last-scattering surface (a_rec ≈ 1/1090) lies at comoving distance D = χ_p(1) − χ_p(a_rec) from us (light after
recombination travels at today's c in every model here). Two points on opposite sides of the sky are 2D apart; their
past light cones overlap only if χ_p(a_rec) ≥ D. Otherwise one horizon spans an angle θ = χ_p(a_rec)/D and the sky
holds ≈ 2/(1 − cos θ) causally disconnected caps. Three ways to fill the gap, batched over models/frames m:

  ΛCDM:      c = c₀, standard H: θ ≈ 1°, the horizon problem.
  inflation: H = H(a_*) (de Sitter) for N e-folds before a_*: r_H shrinks ∝ 1/a, χ_p grows by ≈ c/(a_* H_*) e^N.
  VSL:       c(a) = c₀ [1 + (r − 1) s(a)], s a smooth step of width w e-folds ending at a_* (Albrecht–Magueijo;
             Moffat): a radiation era with c larger by r adds χ ≈ r c₀ a_*/(H₀√Ω_r) · πw/sin(πw).
"""

import math

import torch

from .tensors import A_REC, C0, H0, OMEGA_L, OMEGA_M, OMEGA_R, STEP_WIDTH

# Observational bounds on varying "constants" (dimensionless α = e²/(4πε₀ħc) and the speed of gravity).
BOUNDS = {
    "atomic clocks, α̇/α today (1σ)": {
        "value": (-0.1e-18, 2.1e-18), "unit": "per year", "ref": "Lange et al. 2021, PRL 126, 011102"},
    "Oklo natural reactor (≈2 Gyr ago)": {
        "value": (-0.9e-7, 1.2e-7), "unit": "Δα/α", "ref": "Damour & Dyson 1996"},
    "CMB at z ≈ 1100 (about 0.4 %)": {
        "value": (-4e-3, 4e-3), "unit": "Δα/α", "ref": "Planck intermediate results XXIV (2015)"},
    "GW170817 speed of gravity": {
        "value": (-3e-15, 7e-16), "unit": "(v_gw − c)/c", "ref": "Abbott et al. 2017, ApJL 848, L13"},
}  # fmt: skip


def hubble(a: torch.Tensor) -> torch.Tensor:
    """Standard ΛCDM expansion rate H(a) in km/s/Mpc, elementwise."""
    return H0 * torch.sqrt(OMEGA_R * a**-4 + OMEGA_M * a**-3 + OMEGA_L)


def hubble_with_inflation(a: torch.Tensor, efolds: torch.Tensor, a_star: float) -> torch.Tensor:
    """H frozen at H(a_*) for a_* e^{−N} < a < a_*, ΛCDM elsewhere.   a (A,), efolds (M,) → (M, A)"""
    h_star = hubble(torch.tensor(a_star, dtype=a.dtype, device=a.device))
    inside = (a[None, :] < a_star) & (a[None, :] > a_star * torch.exp(-efolds[:, None]))
    return torch.where(inside, h_star, hubble(a)[None, :].expand(len(efolds), -1))


def speed_of_light(a: torch.Tensor, c_ratio: torch.Tensor, a_star: float) -> torch.Tensor:
    """c(a) = c₀ [1 + (r − 1) σ((ln a_* − ln a)/w)]: r times today's c long before a_*.   (A,), (M,) → (M, A)"""
    step = torch.sigmoid((math.log(a_star) - torch.log(a)) / STEP_WIDTH)  # 1 early, 0 late
    return C0 * (1 + (c_ratio[:, None] - 1) * step[None, :])


def comoving_hubble_radius(a: torch.Tensor, c: torch.Tensor, H: torch.Tensor) -> torch.Tensor:
    """r_H = c / (a H) in Mpc: how far light gets in one expansion time.   broadcast → (M, A)"""
    return c / (a * H)


def particle_horizon(x: torch.Tensor, r_H: torch.Tensor) -> torch.Tensor:
    r"""χ_p(x) = ∫_{x_min}^{x} r_H dx′ (cumulative trapezoid along the last axis).   (M, A) → (M, A)"""
    pieces = 0.5 * (r_H[:, 1:] + r_H[:, :-1]) * x.diff()
    return torch.cat([torch.zeros_like(r_H[:, :1]), pieces.cumsum(-1)], dim=-1)


def horizon_angle(chi_rec: torch.Tensor, distance: torch.Tensor) -> torch.Tensor:
    """θ = χ_p(a_rec)/D, the angle one causal patch subtends on today's sky (capped at π: whole sky)."""
    return (chi_rec / distance).clamp(max=math.pi)


def patch_count(theta: torch.Tensor) -> torch.Tensor:
    """Number of caps of angular radius θ needed to tile the sphere: 4π / (2π(1 − cos θ)), at least 1."""
    return (2 / (1 - torch.cos(theta))).clamp(min=1.0)


def vsl_extra_horizon(c_ratio: torch.Tensor, a_star: float, w: float = STEP_WIDTH) -> torch.Tensor:
    r"""Closed form for the extra horizon from the fast-light era, radiation domination 1/(aH) = a/(H₀√Ω_r):

    (r − 1) c₀/(H₀√Ω_r) ∫ e^x σ((x_* − x)/w) dx = (r − 1) c₀ a_*/(H₀√Ω_r) · πw / sin(πw)   (valid for w < 1).
    """
    return (c_ratio - 1) * C0 * a_star / (H0 * math.sqrt(OMEGA_R)) * (math.pi * w / math.sin(math.pi * w))


def recombination_index(x: torch.Tensor) -> int:
    return int(torch.searchsorted(x, torch.tensor([math.log(A_REC)], dtype=x.dtype, device=x.device)))


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/cosmology/variable_speed_of_light/tensors.py
"""Cosmological parameters (Planck 2018 ΛCDM) and the grid in x = ln a.

Units: Mpc for comoving lengths, km/s for speeds, km/s/Mpc for H, so c/H is in Mpc.
"""

import math

import torch

C0 = 299_792.458  # speed of light today, km/s
H0 = 67.4  # Hubble constant, km/s/Mpc (Planck 2018)
OMEGA_M = 0.315  # matter
OMEGA_R = 9.1e-5  # radiation: photons + 3 massless neutrinos (Ω_r h² ≈ 4.15e-5)
OMEGA_L = 1 - OMEGA_M - OMEGA_R  # dark energy (flat universe)
A_REC = 1 / 1090  # scale factor at last scattering (z ≈ 1089)
STEP_WIDTH = 0.1  # e-folds over which c switches from its early to its present value


def log_a_grid(a_min: float, n: int, device: torch.device) -> torch.Tensor:
    """x = ln a from a_min to today (a = 1), uniform in x.   (A,)"""
    return torch.linspace(math.log(a_min), 0.0, n, dtype=torch.float64, device=device)


def sweep(final: float, n_frames: int, device: torch.device) -> torch.Tensor:
    """Frame-by-frame values 0 … final (the animation turns a model on gradually).   (F,)"""
    return torch.linspace(0.0, final, n_frames, dtype=torch.float64, device=device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/cosmology/variable_speed_of_light/solve.py
"""Horizons for a batch of models at once: every row m of the (M, A) tensors is one universe."""

import math

import torch

from .equation import (
    comoving_hubble_radius,
    horizon_angle,
    hubble,
    hubble_with_inflation,
    particle_horizon,
    patch_count,
    recombination_index,
    speed_of_light,
    vsl_extra_horizon,
)
from .tensors import C0, STEP_WIDTH


def horizons(x: torch.Tensor, model: str, c_ratio=None, efolds=None, a_star: float = 1e-28) -> dict:
    """x = ln a grid (A,). model: "lcdm" (one row), "vsl" (one row per c_ratio), "inflation" (one per efolds).

    Returns c (M, A), H (M, A), the comoving Hubble radius r_H (M, A) and the particle horizon χ_p (M, A), in Mpc.
    """
    a = torch.exp(x)
    if model == "vsl":
        c = speed_of_light(a, c_ratio, a_star)
        H = hubble(a)[None, :].expand_as(c)
    elif model == "inflation":
        H = hubble_with_inflation(a, efolds, a_star)
        c = torch.full_like(H, C0)
    else:
        H = hubble(a)[None, :]
        c = torch.full_like(H, C0)
    r_H = comoving_hubble_radius(a, c, H)
    return {"c": c, "H": H, "hubble_radius": r_H, "chi": particle_horizon(x, r_H)}


def sky_statistics(x: torch.Tensor, h: dict) -> dict:
    """How much of the last-scattering sphere one causal patch covers, for every row.   → each (M,)"""
    k = recombination_index(x)
    chi_rec = h["chi"][:, k]
    distance = h["chi"][:, -1] - chi_rec  # comoving distance to last scattering (c = c₀ after recombination)
    theta = horizon_angle(chi_rec, distance)
    return {"chi_rec": chi_rec, "distance": distance, "theta": theta, "patches": patch_count(theta),
            "solved": chi_rec >= distance}  # fmt: skip


def _gap(x: torch.Tensor) -> tuple[float, float]:
    """(D − χ_p(a_rec), D) for standard ΛCDM: how much horizon is missing."""
    s = sky_statistics(x, horizons(x, "lcdm"))
    return float(s["distance"][0] - s["chi_rec"][0]), float(s["distance"][0])


def vsl_minimum_ratio(x: torch.Tensor, a_star: float) -> float:
    """Smallest c_early/c₀ for which the antipodal past light cones of the CMB overlap (closed-form extra horizon)."""
    missing, _ = _gap(x)
    per_unit = float(vsl_extra_horizon(torch.tensor([2.0], dtype=x.dtype), a_star, STEP_WIDTH))  # (r − 1) = 1
    return 1 + missing / per_unit


def inflation_minimum_efolds(x: torch.Tensor, a_star: float) -> float:
    """Smallest N with c₀/(a_* H_*) (e^N − 1) ≥ missing horizon."""
    missing, _ = _gap(x)
    h_star = float(hubble(torch.tensor(a_star, dtype=x.dtype)))
    return math.log(1 + missing * a_star * h_star / C0)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/cosmology/variable_speed_of_light/plot.py
"""Animate the horizons growing as VSL / inflation are switched on, next to the conformal (light-cone) diagram."""

import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import save_gif

from .tensors import A_REC

GREY, BLUE, GREEN = "#6b7280", "#2563eb", "#16a34a"


def animate(x, std, vsl, infl, stats, c_ratio, efolds, a_star, path: Path) -> Path:
    every = max(1, len(x) // 1200)  # thin the curves for drawing
    la = (x[::every] / math.log(10)).cpu().numpy()  # log10 a
    lg = lambda t: np.log10(np.maximum(t[:, ::every].cpu().numpy(), 1e-300))  # noqa: E731  (χ_p starts at 0)
    rH_s, chi_s = lg(std["hubble_radius"]), lg(std["chi"])
    rH_v, chi_v = lg(vsl["hubble_radius"]), lg(vsl["chi"])
    rH_i, chi_i = lg(infl["hubble_radius"]), lg(infl["chi"])
    s_std, s_vsl, s_inf = ({k: v.cpu().numpy() for k, v in s.items()} for s in stats)
    D = s_std["distance"][0]
    r, N = c_ratio.cpu().numpy(), efolds.cpu().numpy()

    fig, (ax, ax_c) = plt.subplots(1, 2, figsize=(9.6, 4.4), gridspec_kw={"width_ratios": [1.45, 1]})
    ax.axhline(np.log10(D), color="#f59e0b", lw=1.2, ls="--")
    ax.text(-24, np.log10(D) - 3, "distance to the CMB surface D", color="#b45309", fontsize=7)
    ax.axvline(np.log10(A_REC), color="#9ca3af", lw=1, ls=":")
    ax.text(np.log10(A_REC) - 1, -33, "recombination", rotation=90, fontsize=7, color=GREY, ha="right")
    ax.axvline(np.log10(a_star), color="#9ca3af", lw=1, ls=":")
    ax.text(np.log10(a_star) + 1, -33, "$a_*$", fontsize=8, color=GREY)
    ax.plot(la, rH_s[0], color=GREY, lw=1.2, label="ΛCDM  c/(aH)")
    ax.plot(la, chi_s[0], color=GREY, lw=1.2, ls="--", label="ΛCDM  horizon $\\chi_p$")
    l_rv = ax.plot([], [], color=BLUE, lw=1.6, label="VSL  c(a)/(aH)")[0]
    l_cv = ax.plot([], [], color=BLUE, lw=1.6, ls="--", label="VSL  horizon")[0]
    l_ri = ax.plot([], [], color=GREEN, lw=1.3, label="inflation  c/(aH)")[0]
    l_ci = ax.plot([], [], color=GREEN, lw=1.3, ls="--", label="inflation  horizon")[0]
    ax.set_xlim(la[0], 0)
    ax.set_ylim(-36, 14)
    ax.set_xlabel("$\\log_{10} a$")
    ax.set_ylabel("$\\log_{10}$ comoving length (Mpc)")
    ax.legend(fontsize=6.5, loc="lower right", ncol=1)
    title = ax.set_title("placeholder", fontsize=9)

    # conformal diagram: light moves at 45°; lengths in units of D, recombination at η = 0
    ax_c.set_xlim(-1.7, 1.7)
    ax_c.set_ylim(-2.3, 1.15)
    ax_c.set_aspect("equal")
    ax_c.set_xlabel("comoving distance / D")
    ax_c.set_ylabel("conformal time $(\\eta - \\eta_{rec})$ / D")
    ax_c.plot([-1, 0, 1], [0, 1, 0], color="#f59e0b", lw=1.5)  # our past light cone back to last scattering
    ax_c.plot([0], [1], "*", ms=11, color="#111827")
    ax_c.text(0.08, 1.0, "us", fontsize=8)
    ax_c.plot([-1, 1], [0, 0], "o", color="#dc2626", ms=6)
    ax_c.axhline(0, color="#dc2626", lw=0.8, ls=":")
    ax_c.text(1.05, 0.05, "CMB", color="#dc2626", fontsize=8)
    ax_c.grid(False)
    fills = []
    bang_lines = {
        key: ax_c.plot([], [], color=col, lw=2)[0] for key, col in (("std", GREY), ("vsl", BLUE), ("inf", GREEN))
    }
    c_title = ax_c.set_title("placeholder\nplaceholder", fontsize=8)
    bottom = fig.text(0.5, 0.01, "", ha="center", fontsize=8.5)

    def cone_depth(s, f):
        return min(s["chi_rec"][f] / D, 2.25)

    def update(f: int):
        l_rv.set_data(la, rH_v[f])
        l_cv.set_data(la, chi_v[f])
        l_ri.set_data(la, rH_i[f])
        l_ci.set_data(la, chi_i[f])
        title.set_text(f"VSL: $c_{{early}}/c_0 = 10^{{{np.log10(r[f]):.1f}}}$     inflation: N = {N[f]:.1f} e-folds")
        for p in fills:
            p.remove()
        fills.clear()
        for key, s, col in (("vsl", s_vsl, BLUE), ("inf", s_inf, GREEN), ("std", s_std, GREY)):
            k = 0 if key == "std" else f
            d = cone_depth(s, k)
            for side in (-1, 1):  # past light cones of the two antipodal CMB points
                fills.append(ax_c.fill([side - d, side, side + d], [-d, 0, -d], color=col, alpha=0.18, lw=0)[0])
            bang_lines[key].set_data([-1.7, 1.7], [-d, -d] if d < 2.25 else [np.nan, np.nan])
        solved = [name for name, s in (("VSL", s_vsl), ("inflation", s_inf)) if s["solved"][f]]
        verdict = "overlap for " + " and ".join(solved) if solved else "do not overlap: horizon problem"
        c_title.set_text("past light cones of opposite CMB points\n" + verdict)
        bottom.set_text(f"causally disconnected patches on the CMB sky:  ΛCDM {s_std['patches'][0]:,.0f}   "
                        f"VSL {s_vsl['patches'][f]:,.0f}   inflation {s_inf['patches'][f]:,.0f}")  # fmt: skip

    fig.tight_layout(rect=(0, 0.04, 1, 1))
    return save_gif(fig, update, list(range(len(r))) + [len(r) - 1] * 12, path, fps=12)  # hold the end


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'log10_c_ratio': 30.0,
    'efolds': 65.0,
    'log10_a_star': -28.0,
    'n_grid': 40001
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.cosmology.variable_speed_of_light")]:
    del sys.modules[name]
main = importlib.import_module("problems.cosmology.variable_speed_of_light.main")
problem_dir = Path("problems/cosmology/variable_speed_of_light")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
